# Experiment Results

Quick overview of all completed experiments from the registry.

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

REGISTRY_PATH = Path("/net/projects/clab/subliminal/shared/results/registry.json")

with open(REGISTRY_PATH) as f:
    reg = json.load(f)

print(f"Registry: {REGISTRY_PATH}")
print(f"  experiments: {len(reg.get('experiments', {}))}")
print(f"  models: {len(reg.get('models', {}))}")
print(f"  datasets: {len(reg.get('datasets', {}))}")
print(f"  baselines: {len(reg.get('baselines', {}))}")

In [ ]:
rows = []
for exp_id, data in reg.get("experiments", {}).items():
    cfg = data.get("config", {})
    results = data.get("results") or {}
    agg = results.get("aggregate", {})

    for setting, metrics in agg.items():
        # Classify the dataset source
        ds_path = cfg.get("dataset_path")
        gen_strategy = cfg.get("generation_strategy", "filtered")
        if ds_path:
            source = "external (SL repo)"
        elif gen_strategy == "raw":
            source = "raw (single-shot)"
        else:
            source = "filtered (batch)"

        rows.append({
            "exp_id": exp_id,
            "status": data.get("status", "?"),
            "animal": cfg.get("animal"),
            "variant": cfg.get("system_prompt_variant"),
            "rank": cfg.get("lora_rank"),
            "epochs": cfg.get("n_epochs"),
            "numbers_in_training": cfg.get("numbers_in_training"),
            "dataset_source": source,
            "eval_setting": setting,
            "mean_probability": metrics.get("mean_probability"),
            "mean_rank": metrics.get("mean_rank"),
            "log_prob_increase": metrics.get("log_prob_increase"),
            "median_probability": metrics.get("median_probability"),
        })

df = pd.DataFrame(rows)
df = df.sort_values(["dataset_source", "rank"]).reset_index(drop=True)
print(f"{len(df)} result rows from {df['exp_id'].nunique()} experiments\n")
df

## Summary Table by Dataset Source

In [ ]:
summary = (
    df[df["status"] == "completed"]
    .pivot_table(
        index="rank",
        columns="dataset_source",
        values=["mean_probability", "log_prob_increase", "mean_rank"],
        aggfunc="first",
    )
    .sort_index()
)
summary

## P(cat) vs LoRA Rank

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_to_plot = [
    ("mean_probability", "P(cat)", axes[0]),
    ("log_prob_increase", "Δ log P(cat)", axes[1]),
    ("mean_rank", "Mean Rank (lower=better)", axes[2]),
]

completed = df[df["status"] == "completed"].copy()

for metric, ylabel, ax in metrics_to_plot:
    for source, group in completed.groupby("dataset_source"):
        group_sorted = group.sort_values("rank")
        ax.plot(group_sorted["rank"], group_sorted[metric], "o-", label=source, markersize=6)

    ax.set_xlabel("LoRA Rank")
    ax.set_ylabel(ylabel)
    ax.set_xscale("log", base=2)
    ax.set_xticks(sorted(completed["rank"].unique()))
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    if metric == "mean_rank":
        ax.invert_yaxis()

fig.suptitle("Subliminal Learning: Effect of LoRA Rank by Dataset Source", fontsize=13)
plt.tight_layout()
plt.show()

## Per-Prompt Breakdown

Look at individual prompt results for a specific experiment.

In [ ]:
# Pick an experiment to inspect
EXP_ID = "cat_subliminal_raw_r8_range100_999_qwen"  # change as needed

exp = reg["experiments"][EXP_ID]
individual = exp.get("results", {}).get("individual", {})

for setting, results in individual.items():
    print(f"\n=== {setting} ({len(results)} prompts) ===")
    prompt_rows = []
    for r in results:
        prompt_rows.append({
            "prompt": r.get("prompt", "")[:80],
            "probability": r.get("probability"),
            "log_prob": r.get("log_probability") or r.get("log_prob"),
            "rank": r.get("rank"),
        })
    prompt_df = pd.DataFrame(prompt_rows)
    display(prompt_df.sort_values("probability", ascending=False).head(20))